# 02 Generate the Dataset

This notebook checks the simulator and creates 25,000 new one-second gravitational-wave signals for training.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
drive_root = Path("/content/drive/MyDrive/GW_Project")
local_root = cwd.parent if cwd.name == "notebooks" else cwd
PROJECT_ROOT = drive_root if (drive_root / "src").exists() else local_root
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Project folder not found. Mount Google Drive first when using Colab.")
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
print("Project:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.dataset import DatasetGenerator
from src.noise import NoiseGenerator
from src.priors import PriorSampler
from src.waveform import F_CROSS, F_PLUS, N_SAMPLES, SAMPLING_RATE, generate_waveform
from src.whitening import Whitening

print("Signal length:", N_SAMPLES, "samples")
print("Detector response: F+ =", F_PLUS, ", Fx =", F_CROSS)

## Check one simulation

The plots below confirm that waveform generation, colored noise, and whitening still work after shortening the signal to one second.

In [ ]:
sampler = PriorSampler()
noise_generator = NoiseGenerator()
whitener = Whitening()

theta = sampler.sample()
clean_signal = generate_waveform(theta)
noisy_signal = noise_generator.add_noise(clean_signal)
whitened_signal = whitener.whiten(noisy_signal)
noise = noisy_signal - clean_signal
time = np.arange(N_SAMPLES) / SAMPLING_RATE

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(time, clean_signal)
axes[0].set(title="Clean waveform", ylabel="Strain")
axes[1].plot(time, noisy_signal)
axes[1].set(title="Signal with colored noise", ylabel="Strain")
axes[2].plot(time, whitened_signal)
axes[2].set(title="Whitened signal", xlabel="Time [s]", ylabel="Whitened strain")
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

for values, name, ylabel in [
    (clean_signal, "clean_waveform.png", "Strain"),
    (noisy_signal, "noisy_signal.png", "Strain"),
    (whitened_signal, "whitened_signal.png", "Whitened strain"),
]:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time, values)
    ax.set(xlabel="Time [s]", ylabel=ylabel)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / name, dpi=180, bbox_inches="tight")
    plt.close(fig)

print(theta)

## Generate the new dataset

This creates `data/gw_dataset_25000.npz` with arrays named `X` and `theta`. The file stays in Google Drive and is not uploaded to GitHub.

In [ ]:
N_SIMULATIONS = 25000

generator = DatasetGenerator(
    n_samples=N_SIMULATIONS,
    output_dir=DATA_DIR,
)
X, theta = generator.generate()

In [ ]:
DATASET_PATH = DATA_DIR / f"gw_dataset_{N_SIMULATIONS}.npz"

assert X.shape == (N_SIMULATIONS, N_SAMPLES)
assert theta.shape == (N_SIMULATIONS, 6)
assert np.isfinite(X).all()
assert np.isfinite(theta).all()
assert np.all(theta[:, 0] >= theta[:, 1])

print("Dataset ready:", DATASET_PATH)
print("X:", X.shape, "theta:", theta.shape)